# Vehicle detection using convolutional neural network for autonomous driving facility

Dataset: https://www.kaggle.com/datasets/sshikamaru/car-object-detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# importing neccesary liabraries

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import os
import random

#Loading the data

In [ ]:
# Calculate the number of images in the training and testing directories
number_of_training_images = len(os.listdir('/content/drive/MyDrive/data/training_images'))
number_of_test_images = len(os.listdir('/content/drive/MyDrive/data/testing_images'))

# Display the counts of training and test images
print('Count of Training Images:', number_of_training_images)
print('Count of Test Images:', number_of_test_images)

In [ ]:
Car_Data=pd.read_csv('/content/drive/MyDrive/data/train_solution_bounding_boxes (1).csv')
Car_Data.head()

In [ ]:
len_data = len(Car_Data)
print('Number of Train data localization:', len_data)

In [ ]:
# Iterate through each entry in the Car_Data values
for entry in Car_Data.values:
    photo = plt.imread(f'/content/drive/MyDrive/data/training_images/{entry[0]}')

    photo = photo.copy()

    plt.imshow(photo)
    print('Shape of Image:', photo.shape)

    print('Photo Details - Name, xmin, ymin, xmax, ymax:', entry)

    top_left = (int(entry[1]), int(entry[2]))
    bottom_right = (int(entry[3]), int(entry[4]))

    cv2.rectangle(photo, top_left, bottom_right, (0, 255, 0), 2)

    plt.figure()
    plt.imshow(photo)

    break


In [ ]:
# Iterate through the enumerated values in the Data
for index, entry in enumerate(Car_Data.values):
    image = plt.imread('/content/drive/MyDrive/data/training_images/' + entry[0])
    image=image.copy()
    plt.figure()
    plt.imshow(image)
    xmin = int(entry[1])
    ymin = int(entry[2])
    xmax = int(entry[3])
    ymax = int(entry[4])
    cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
    plt.figure()
    plt.imshow(image)

    if index == 2:
        break


# Selective search

In [ ]:
cv2.setUseOptimized(True)
ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()

In [ ]:
# Read an image and resize it to dimensions 224x224
image = cv2.imread('/content/drive/MyDrive/data/training_images/vid_4_1000.jpg')
image = cv2.resize(image, (224, 224))

# Display the resized image
plt.figure()
plt.imshow(image)

# Initialize Selective Search Segmentation with the resized image
ss = cv2.ximgproc.segmentation.createSelectiveSearchSegmentation()
ss.setBaseImage(image)
ss.switchToSelectiveSearchFast()

# Get a list of possible bounding boxes using the segmentation
bounding_boxes = ss.process()
print('Number of possible bounding boxes:', len(bounding_boxes))

# Draw rectangles around the detected regions
for rect in bounding_boxes:
    x, y, w, h = rect
    imOut = cv2.rectangle(image, (x, y), (x+w, y+h), (0, 255, 0), 1, cv2.LINE_AA)

# Display the image with drawn bounding boxes
plt.figure()
plt.imshow(imOut)


In [ ]:
def calculate_iou(bbox1, bbox2):
    assert bbox1['left'] < bbox1['right'] and bbox1['top'] < bbox1['bottom']
    assert bbox2['left'] < bbox2['right'] and bbox2['top'] < bbox2['bottom']

    overlap_left = max(bbox1['left'], bbox2['left'])
    overlap_top = max(bbox1['top'], bbox2['top'])
    overlap_right = min(bbox1['right'], bbox2['right'])
    overlap_bottom = min(bbox1['bottom'], bbox2['bottom'])

    if overlap_right < overlap_left or overlap_bottom < overlap_top:
        return 0.0

    overlap_area = (overlap_right - overlap_left) * (overlap_bottom - overlap_top)
    bbox1_area = (bbox1['right'] - bbox1['left']) * (bbox1['bottom'] - bbox1['top'])
    bbox2_area = (bbox2['right'] - bbox2['left']) * (bbox2['bottom'] - bbox2['top'])

    iou = overlap_area / float(bbox1_area + bbox2_area - overlap_area)
    assert 0.0 <= iou <= 1.0

    return iou

In [ ]:
# Initialize an empty list to store processed image data and labels
processed_images = []

# Initialize counters for positive and negative samples
positive_samples = 0
negative_samples = 0
iterations = 0

# Iterate through rows in the CSV file
for data_row in pd.read_csv('/content/drive/MyDrive/data/train_solution_bounding_boxes (1).csv').values:
    image_name, x_min, y_min, x_max, y_max = data_row

    # Create a dictionary for the ground truth bounding box (ground_truth_box)
    ground_truth_box = {
        'left': int(x_min),
        'top': int(y_min),
        'right': int(x_max),
        'bottom': int(y_max)
    }

    try:
        # Read the image using the provided filename
        image = cv2.imread('/content/drive/MyDrive/data/training_images/' + image_name)

        # Set up selective search
        ss.setBaseImage(image)
        ss.switchToSelectiveSearchFast()

        # Iterate through proposed regions from selective search
        for proposal_box in ss.process():
            x_prop, y_prop, w_prop, h_prop = proposal_box

            # Create a dictionary for the proposed bounding box (proposed_box)
            proposed_box = {
                'left': x_prop,
                'top': y_prop,
                'right': x_prop + w_prop,
                'bottom': y_prop + h_prop
            }

            # Determine if the number of positive samples is below a threshold
            if positive_samples < negative_samples:
                iou = calculate_iou(ground_truth_box, proposed_box)
                if 0.5 < iou:
                    processed_images.append([cv2.resize(image[proposed_box['top']:proposed_box['bottom'], proposed_box['left']:proposed_box['right']], (224, 224)), 1])
                    positive_samples += 1
            else:
                iou = calculate_iou(ground_truth_box, proposed_box)
                if 0.5 < iou:
                    processed_images.append([cv2.resize(image[proposed_box['top']:proposed_box['bottom'], proposed_box['left']:proposed_box['right']], (224, 224)), 1])
                    positive_samples += 1
                else:
                    processed_images.append([cv2.resize(image[proposed_box['top']:proposed_box['bottom'], proposed_box['left']:proposed_box['right']], (224, 224)), 0])
                    negative_samples += 1
    except Exception as error:
        print('Error:', error)

    iterations += 1
    print('Iteration:', iterations, 'Processed Proposals:', len(ss.process()))


In [ ]:
print(len(processed_images))

In [ ]:
data=[]
data_label=[]
for features,label in processed_images:
  data.append(features)
  data_label.append(label)
print('success')

In [ ]:
modified_data = []
modified_data_label = []

for modified_features, modified_label in processed_images:
    modified_data.append(modified_features)
    modified_data_label.append(modified_label)

print('Extraction successful')


In [ ]:
print('Number of Photos:',len(modified_data),'|Number of Labels:',len(modified_data_label))

In [ ]:
i=random.randint(1,10583)
print('Class:',modified_data_label[i])
print('Shape:',modified_data[i].shape)
plt.imshow(data[i]);

In [ ]:
modified_data=np.asarray(modified_data)
modified_data_label=np.asarray(modified_data_label)

In [ ]:
len_no_car_image = len(modified_data_label[modified_data_label==0])
len_car_image = len(modified_data_label[modified_data_label==1])
print('No car Image:', len_no_car_image,'|Car Image:', len_car_image)

In [ ]:
modified_data.shape
modified_data_label.shape

In [ ]:
from sklearn.model_selection import train_test_split

# Split the modified_data and modified_data_label
x_train, x_val, y_train, y_val = train_test_split(modified_data, modified_data_label, test_size=0.40, random_state=42)

print('x_train shape:', x_train.shape)
print('x_val shape:', x_val.shape)
print('y_train shape:', y_train.shape)
print('y_val shape:', y_val.shape)


#Code

In [ ]:
og_model=tf.keras.applications.VGG16(include_top=False,input_shape=(224,224,3),weights='imagenet')
og_model.summary()

In [ ]:
car_model=tf.keras.Sequential()
car_model.add(og_model)
car_model.add(tf.keras.layers.GlobalAveragePooling2D())
car_model.add(tf.keras.layers.Dropout(0.5))
car_model.add(tf.keras.layers.Dense(1,activation='sigmoid'))


In [ ]:
car_model.trainable=False

In [ ]:
for i,layer in enumerate(og_model.layers):
  print(i,layer.name,'-',layer.trainable)

In [ ]:
car_model.compile(loss='binary_crossentropy',optimizer=tf.keras.optimizers.Adam(),metrics=['accuracy'])

In [ ]:
epoch=5
result=car_model.fit(x_train,y_train,epochs=epoch,validation_data=(x_val,y_val))

# Test

In [ ]:
car_list = []
image_url = '/content/drive/MyDrive/data/testing_images/vid_5_27620.jpg'  # Change the path for different inputs
test_image = cv2.imread(image_url)
ss.setBaseImage(test_image)
ss.switchToSelectiveSearchFast()
print('Number of potential objects in the Photo:', len(ss.process()))

for proposal in ss.process():
    x, y, w, h = proposal
    bounding_box = {
        'x1': x,
        'y1': y,
        'x2': x + w,
        'y2': y + h
    }

    try:
        assert bounding_box['x1'] < bounding_box['x2']
        assert bounding_box['y1'] < bounding_box['y2']

        img_data = test_image[bounding_box['y1']:bounding_box['y2'], bounding_box['x1']:bounding_box['x2']]
        img_data = cv2.resize(img_data, (224, 224))

        prediction = car_model.predict(img_data.reshape(1, 224, 224, 3))[0]

        if prediction > 0.5:
            car_list.append([bounding_box, prediction])
        else:
            pass
    except Exception as err:
        print('Error:', err)

print('Number of bounding boxes where class = 1:', len(car_list))
test_image = cv2.imread(image_url)
highest_score_box = car_list[np.argmax(np.array(car_list)[:, 1])][0]
pt1 = (highest_score_box['x1'], highest_score_box['y1'])
pt2 = (highest_score_box['x2'], highest_score_box['y2'])
plt.figure()
plt.imshow(test_image)
cv2.rectangle(test_image, pt1, pt2, (255, 0, 0), 2)
plt.figure()
plt.title(f'Highest Scoring Bounding Box with Class = 1: %{car_list[np.argmax(np.array(car_list)[:, 1])][1][0]*100}')
plt.imshow(test_image)
